# Representation similarity per period (all optimizers)

For every `(experiment_id, optimizer_id)` in [`scan_results()`](../app.py), compute **`repr_per_period`** only — the same aggregation as in [`compare_all_optimizers.ipynb`](compare_all_optimizers.ipynb) (mean `representation_similarity_factor` over checkpoints, grouped by `neuron_id` / `period_id`).

Runs are limited to pairs whose **`mean_total_score`** from the full scoring pipeline in `compare_all_optimizers` is **> 10** (loaded from that notebook’s `SAVE_DIR`, default `local/save/COMP/`). Run that notebook first so those checkpoints exist.

Checkpoints are written under `SAVE_DIR` (see below). Through-time representation factors use **`get_period_repr_factors_through_time`** in this notebook: truncated Jacobian saliency (only hidden layers up to the deepest row at each checkpoint), then rows grouped by ``layer_name`` — same keys/values as the full ``get_all_saliency_maps`` path, without per-digit MSSIM.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gzip
import hashlib
import pickle
import sys
from pathlib import Path
from typing import Any, cast

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


def _find_project_root(start: Path | None = None) -> Path:
	p = (start or Path.cwd()).resolve()
	for cand in [p, *p.parents]:
		if (cand / "pyproject.toml").is_file():
			return cand
	return p


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
	sys.path.insert(0, str(PROJECT_ROOT))

from experiments.mnist import MNISTWrapper
from models import get_model
from models.unit_node_id import parse_unit_node_id

from visualize_webapp.app import (
	_NETWORK_SAMPLES_PER_DIGIT,
	_metrics,
	_nts,
	_sort_optimizer_ids,
	scan_results,
)
from visualize_webapp.notebook.scoring_constants import USE_CW_SSIM
from visualize_webapp.notebook.scoring_helpers import (
	_all_custom_saliency_maps_from_weights_and_activations,
	_dnn_hidden_layer_names_from_nts,
	_stack_saliency_maps_for_mssim,
	_unit_key,
	compute_repr_sim_factor,
	filter_periods,
	find_impacted_periods,
	get_all_model_saliency_maps,
	get_cross_sample_distances_from_maps,
	get_data_range,
	get_iteration,
	get_neuron_saliency_maps_from_all,
	get_n_assigned_digits,
	get_period_checkpoint_indices,
	get_recovery_periods,
	init_cw_ssim_pyramid,
	remove_inelligible_periods,
	index_grp,
)
from visualize_webapp.post_processing import _pp_dead, _pp_neuron_digit

EXPERT_MODEL_DIR = "local/save/expert_optimizer_pretrained/!OPT/"
STRICT = False
BTSP_START_STRATEGY = "acceleration"
SAVE_DIR = "local/save/REPR/"
# Must match ``SAVE_DIR`` in ``compare_all_optimizers.ipynb`` (``score_run`` / ``mean_total_score``).
COMPARE_ALL_OPTIMIZERS_SAVE_DIR = "local/save/COMP/"
MIN_MEAN_TOTAL_SCORE = 10.0


def _run_checkpoint_dir(save_dir: str, eid: str, oid: str) -> Path:
	sha = hashlib.sha256(f"{eid}-{oid}".encode("utf-8")).hexdigest()[:7]
	return Path(save_dir) / sha


def _save_run_checkpoint(save_dir: str, eid: str, oid: str, out: dict[str, Any]) -> None:
	d = _run_checkpoint_dir(save_dir, eid, oid)
	d.mkdir(parents=True, exist_ok=True)
	pd.DataFrame([{"eid": eid, "oid": oid}]).to_csv(d / "key.csv", index=False)
	with gzip.open(d / "results.pkl.gz", "wb", compresslevel=9) as f:
		pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)


def _load_run_checkpoint(save_dir: str, eid: str, oid: str) -> dict[str, Any] | None:
	d = _run_checkpoint_dir(save_dir, eid, oid)
	key_csv = d / "key.csv"
	res_gz = d / "results.pkl.gz"
	res_pkl = d / "results.pkl"
	if not key_csv.is_file() or (not res_gz.is_file() and not res_pkl.is_file()):
		return None
	key_df = pd.read_csv(key_csv)
	if len(key_df) != 1 or str(key_df["eid"].iloc[0]) != str(eid) or str(key_df["oid"].iloc[0]) != str(oid):
		return None
	if res_gz.is_file():
		with gzip.open(res_gz, "rb") as f:
			return pickle.load(f)
	with open(res_pkl, "rb") as f:
		return pickle.load(f)


def _hidden_layer_depth(layer_name) -> int:
	"""Layer stack index (``hidden.{2k}`` → ``k``); matches ``_layer_index_from_neuron_id``."""
	return int(str(layer_name).split(".")[-1]) // 2


def get_all_saliency_maps_up_to_layer(
	nts,
	checkpoint_idx: int,
	max_layer_idx: int,
	*,
	device=None,
	convolve_: bool = False,
):
	"""Like ``scoring_helpers.get_all_saliency_maps`` but only layers ``0..max_layer_idx``.

	``custom_saliency_map`` builds layer Jacobians in order; skipping tail layers avoids
	work for neurons that only live in earlier hidden blocks.
	"""
	device = torch.device(device) if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
	layer_names_full = _dnn_hidden_layer_names_from_nts(nts)
	if not layer_names_full:
		raise ValueError("No DNN hidden layers found in neuron timeseries")
	max_layer_idx = int(min(max(max_layer_idx, 0), len(layer_names_full) - 1))
	layer_names = layer_names_full[: max_layer_idx + 1]

	weights = []
	activations = []
	for layer_name in layer_names:
		layer_nids_with_indices = []
		for nid in nts["unit_node_ids"]:
			nid = str(nid)
			parsed = parse_unit_node_id(nid)
			if parsed is not None and parsed["layer_name"] == layer_name:
				layer_nids_with_indices.append((parsed["unit_index"], nid))
		layer_nids = [nid for _, nid in sorted(layer_nids_with_indices)]
		weights.append(
			torch.tensor(np.stack([nts[_unit_key("weights", nid)][checkpoint_idx] for nid in layer_nids])).float().to(device)
		)
		activations.append(
			torch.tensor(np.stack([nts[_unit_key("act", nid)][checkpoint_idx] for nid in layer_nids])).float().to(device)
		)

	with torch.no_grad():
		all_saliency_maps = _all_custom_saliency_maps_from_weights_and_activations(
			weights,
			torch.stack(activations),
			convolve_=convolve_,
		).cpu()
	del weights, activations
	if device.type == "cuda":
		torch.cuda.empty_cache()
	return all_saliency_maps


def get_period_repr_factors_through_time(
	nts,
	recovery_periods,
	expert_saliency,
	*,
	device=None,
	convolve_: bool = False,
	metrics=None,
	prebuilt_pyramids: dict | None = None,
	checkpoint_df=None,
	btsp_start_strategy: str = "trial_start",
) -> pd.DataFrame:
	"""Per-checkpoint ``representation_similarity_factor`` only (no MSSIM).

	Copied from the inner loop of ``scoring_helpers.get_period_scores_through_time``,
	without the ``for d in range(_NETWORK_N_DIGITS)`` / ``saliency_digit_pair_mssim`` block.
	Per checkpoint, saliency Jacobians run only through the deepest ``layer_name`` present
	(``get_all_saliency_maps_up_to_layer``); rows are processed grouped by ``layer_name``.
	"""
	pyramids_cache = {} if prebuilt_pyramids is None else prebuilt_pyramids
	if checkpoint_df is None:
		checkpoint_df = get_period_checkpoint_indices(
			recovery_periods, nts=nts, metrics=metrics, btsp_start_strategy=btsp_start_strategy
		)
	if checkpoint_df.empty:
		return pd.DataFrame(
			columns=index_grp + ["checkpoint_idx", "representation_similarity_factor"]
		).set_index(index_grp + ["checkpoint_idx"])

	expert_grid_cache: dict = {}
	repr_records: list[dict] = []

	for checkpoint_idx, checkpoint_rows in tqdm(
		checkpoint_df.groupby("checkpoint_idx", sort=True),
		desc="Computing representation similarity through time",
	):
		checkpoint_idx = int(checkpoint_idx)
		max_layer_idx = int(checkpoint_rows["layer_name"].map(_hidden_layer_depth).max())
		all_saliency_maps = get_all_saliency_maps_up_to_layer(
			nts,
			checkpoint_idx,
			max_layer_idx,
			device=device,
			convolve_=convolve_,
		)

		for layer_name in sorted(
			checkpoint_rows["layer_name"].unique(),
			key=lambda ln: int(str(ln).split(".")[-1]),
		):
			rows_same_layer = checkpoint_rows[checkpoint_rows["layer_name"] == layer_name]
			for row in rows_same_layer.to_dict("records"):
				nid, pid = row["neuron_id"], row["period_id"]
				model_saliency = get_neuron_saliency_maps_from_all(all_saliency_maps, nid)
				model_maps = _stack_saliency_maps_for_mssim(model_saliency, keep_torch=USE_CW_SSIM)

				model_distances = get_cross_sample_distances_from_maps(
					model_maps,
					data_range=get_data_range(model_maps),
					compute_full_grid=True,
					prebuilt_pyramids=pyramids_cache,
				)["cross_sample_distances"]

				if nid not in expert_grid_cache:
					exp_maps = _stack_saliency_maps_for_mssim(
						expert_saliency[nid],
						keep_torch=USE_CW_SSIM,
					)
					expert_grid_cache[nid] = get_cross_sample_distances_from_maps(
						exp_maps,
						data_range=get_data_range(exp_maps),
						compute_full_grid=True,
						prebuilt_pyramids=pyramids_cache,
					)["cross_sample_distances"]

				repr_records.append(
					{
						"neuron_id": nid,
						"period_id": pid,
						"checkpoint_idx": checkpoint_idx,
						"representation_similarity_factor": compute_repr_sim_factor(
							nts,
							nid,
							pid,
							model_distances,
							expert_grid_cache[nid],
							recovery_periods,
						),
					}
				)

				del model_saliency, model_maps, model_distances

		del all_saliency_maps
		if torch.cuda.is_available():
			torch.cuda.empty_cache()

	return pd.DataFrame(repr_records).set_index(index_grp + ["checkpoint_idx"])


class DummyExp(MNISTWrapper):
	def experiment_id(self):
		return ""

	def _build_trials(self, batch_size: int, seed: int, *, num_experiment_runs: int):
		return []


def repr_only_run(
	eid: str,
	mid: str,
	oid: str,
	rid: str,
	prebuilt_pyramids: dict,
	project_root: Path | None = None,
	expert_saliency_by_oid: dict[str, dict] | None = None,
) -> dict[str, Any]:
	"""Match ``compare_all_optimizers`` / ``score_run`` up through ``repr_per_period`` only (no MSSIM)."""
	root = project_root or PROJECT_ROOT
	metrics = _metrics(eid, mid, oid, rid=rid, w_cache=False)
	df_nd = cast(pd.DataFrame, _pp_neuron_digit(eid, mid, oid, rid=rid, w_cache=False))
	if df_nd is None or df_nd.empty:
		return {"error": "empty neuron_digit", "eid": eid, "mid": mid, "oid": oid, "rid": rid}

	df_nd = df_nd.copy()
	df_nd["n_active"] = df_nd["status"].apply(
		lambda s: 0
		if s == "inactive"
		else (_NETWORK_SAMPLES_PER_DIGIT if s == "assigned" else int(s.split("_")[-1]))
	)
	df_nd["iteration"] = df_nd["checkpoint_idx"].map(lambda c: get_iteration(metrics, c))

	df_dead = cast(pd.DataFrame, _pp_dead(eid, mid, oid, rid=rid, w_cache=False))
	if df_dead is not None and not df_dead.empty:
		df_dead = df_dead.copy()
		df_dead["iteration"] = df_dead["checkpoint_idx"].map(lambda c: get_iteration(metrics, c))

	nts = _nts(eid, mid, oid, rid=rid, w_cache=False)

	n_assigned_digits = get_n_assigned_digits(df_nd)
	recovery_periods_any = get_recovery_periods(
		metrics, df_nd, n_assigned_digits, df_dead, mode="assigned_any"
	)
	recovery_periods_same = get_recovery_periods(
		metrics, df_nd, n_assigned_digits, df_dead, mode="assigned_same"
	)
	impacted = find_impacted_periods(recovery_periods_any, recovery_periods_same)
	n_any = len(recovery_periods_any)
	impacted_pct = float(len(impacted) / n_any) if n_any else 0.0

	recovery_periods = recovery_periods_same
	recovery_periods = remove_inelligible_periods(recovery_periods)
	if df_dead is not None and not df_dead.empty:
		recovery_periods = filter_periods(recovery_periods, df_dead, strict=STRICT)

	empty_repr = pd.DataFrame(
		{"representation_similarity_factor": []},
		index=pd.MultiIndex.from_tuples([], names=index_grp),
	)

	if recovery_periods.empty:
		return {
			"eid": eid,
			"mid": mid,
			"oid": oid,
			"rid": rid,
			"impacted_pct": impacted_pct,
			"repr_per_period": empty_repr,
		}

	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	eval_digits = DummyExp().evaluation_inputs(device)[0]

	_, prebuilt_pyramids = init_cw_ssim_pyramid(device, prebuilt_pyramids)

	required_nids = list(recovery_periods["neuron_id"].unique())
	expert_dir = root / EXPERT_MODEL_DIR.replace("!OPT", oid.replace("grafted", "graft")).replace(
		"adam_lr0.01", "adam_lr0.0005"
	)
	expert_saliency: dict
	if expert_saliency_by_oid is not None and oid in expert_saliency_by_oid:
		expert_saliency = expert_saliency_by_oid[oid]
		missing = [nid for nid in required_nids if nid not in expert_saliency]
		if missing:
			expert_meta = torch.load(expert_dir / "train_meta.pt", map_location="cpu", weights_only=True)
			expert_model = get_model(expert_meta["MODEL_TYPE"], **expert_meta["MODEL_CONFIG"]).to(device)
			expert_model.load_state_dict(torch.load(expert_dir / "model.pt", map_location="cpu", weights_only=True))
			expert_model.eval()
			all_expert_saliency_maps = get_all_model_saliency_maps(expert_model, eval_digits, device)
			for nid in tqdm(missing, desc=f"Expert maps (extra) {oid[:16]}…", leave=False):
				expert_saliency[nid] = get_neuron_saliency_maps_from_all(all_expert_saliency_maps, nid)
			expert_model.cpu()
			del expert_model, all_expert_saliency_maps
			if torch.cuda.is_available():
				torch.cuda.empty_cache()
	else:
		expert_meta = torch.load(expert_dir / "train_meta.pt", map_location="cpu", weights_only=True)
		expert_model = get_model(expert_meta["MODEL_TYPE"], **expert_meta["MODEL_CONFIG"]).to(device)
		expert_model.load_state_dict(torch.load(expert_dir / "model.pt", map_location="cpu", weights_only=True))
		expert_model.eval()
		all_expert_saliency_maps = get_all_model_saliency_maps(expert_model, eval_digits, device)
		expert_saliency = {
			nid: get_neuron_saliency_maps_from_all(all_expert_saliency_maps, nid)
			for nid in tqdm(required_nids, desc=f"Expert maps {oid[:16]}…", leave=False)
		}
		expert_model.cpu()
		del expert_model, all_expert_saliency_maps
		if torch.cuda.is_available():
			torch.cuda.empty_cache()
		if expert_saliency_by_oid is not None:
			expert_saliency_by_oid[oid] = expert_saliency

	last_cp = len(metrics["checkpoint_iterations"]) - 1
	checkpoint_df_parts = []
	for _, row in recovery_periods.iterrows():
		end_cp = int(row["end"]) if pd.notna(row["end"]) else last_cp + 1
		start_cp = int(row["start"])
		stride = max(1, (end_cp - start_cp) // 15)
		part = get_period_checkpoint_indices(
			recovery_periods.loc[[row.name]],
			stride=stride,
			nts=nts,
			metrics=metrics,
			btsp_start_strategy=BTSP_START_STRATEGY,
		)
		checkpoint_df_parts.append(part)
	checkpoint_df = pd.concat(checkpoint_df_parts, ignore_index=True)

	df_repr_tt = get_period_repr_factors_through_time(
		nts,
		recovery_periods,
		expert_saliency,
		device=device,
		metrics=metrics,
		prebuilt_pyramids=prebuilt_pyramids,
		checkpoint_df=checkpoint_df,
		btsp_start_strategy=BTSP_START_STRATEGY,
	)

	repr_per_period = (
		df_repr_tt.reset_index()
		.groupby(index_grp)["representation_similarity_factor"]
		.mean()
		.to_frame()
	)

	return {
		"eid": eid,
		"mid": mid,
		"oid": oid,
		"rid": rid,
		"impacted_pct": impacted_pct,
		"repr_per_period": repr_per_period,
	}

In [3]:
tree = scan_results()
prebuilt_pyramids: dict = {}
expert_saliency_by_oid: dict[str, dict] = {}
all_results: dict[tuple[str, str], dict] = {}
errors: list[dict] = []

In [4]:
tree

{'digit_ordered_0_1_75_25_tr100': {'dnn_5x64': {'0d4f4a': ['sgd_lr0.01',
    'adagrad_lr0.01',
    'adam_lr0.01',
    'pure_shampoo_lr0.01',
    'grafted_shampoo_lr0.01']}},
 'pretrainK1000_tr100_then_0': {'dnn_5x64': {'6c35d5': ['sgd_lr0.01',
    'adagrad_lr0.01',
    'adam_lr0.01',
    'pure_shampoo_lr0.01',
    'grafted_shampoo_lr0.01']}},
 'pretrain_0_mislabel1_1_K1000_tr100': {'dnn_5x64': {'4a7338': ['sgd_lr0.01',
    'adagrad_lr0.01',
    'adam_lr0.01',
    'pure_shampoo_lr0.01',
    'grafted_shampoo_lr0.01']}},
 'pretrain_relabelC_0_1_2_K1000_tr100': {'dnn_5x64': {'6cef7f': ['sgd_lr0.01',
    'adagrad_lr0.01',
    'adam_lr0.01',
    'pure_shampoo_lr0.01',
    'grafted_shampoo_lr0.01']}}}

In [ ]:
for eid, models in tqdm(sorted(tree.items()), desc="Experiments"):
	for mid, runs in sorted(models.items()):
		for rid, oids in sorted(runs.items()):
			for oid in _sort_optimizer_ids(list(oids)):
				key = (eid, oid)
				if key in all_results:
					print(f"Skipping {key} because it already exists in memory")
					continue
				scored = _load_run_checkpoint(COMPARE_ALL_OPTIMIZERS_SAVE_DIR, eid, oid)
				mts = scored.get("mean_total_score") if scored else None
				if mts is None or pd.isna(mts) or float(mts) <= MIN_MEAN_TOTAL_SCORE:
					print(
						f"Skipping {key}: mean_total_score={mts!r} "
						f"(need > {MIN_MEAN_TOTAL_SCORE}; load from {COMPARE_ALL_OPTIMIZERS_SAVE_DIR!r})"
					)
					continue
				loaded = _load_run_checkpoint(SAVE_DIR, eid, oid)
				if loaded is not None:
					all_results[key] = loaded
					print(f"Loaded {key} from checkpoint")
					continue
				out = repr_only_run(
					eid, mid, oid, rid, prebuilt_pyramids, PROJECT_ROOT, expert_saliency_by_oid
				)
				_save_run_checkpoint(SAVE_DIR, eid, oid, out)
				all_results[key] = out
				if "error" in out:
					errors.append(out)

print(f"Computed repr for {len(all_results)} (eid, oid) pairs; {len(errors)} errors")
if errors:
	from IPython.display import display

	display(pd.DataFrame(errors))

Experiments:   0%|          | 0/4 [00:00<?, ?it/s]

Skipping ('digit_ordered_0_1_75_25_tr100', 'sgd_lr0.01'): mean_total_score=2.094143601378222 (need > 10.0; load from 'local/save/COMP/')
Skipping ('digit_ordered_0_1_75_25_tr100', 'adagrad_lr0.01'): mean_total_score=2.237706293351449 (need > 10.0; load from 'local/save/COMP/')
Loaded ('digit_ordered_0_1_75_25_tr100', 'adam_lr0.01') from checkpoint
Skipping ('digit_ordered_0_1_75_25_tr100', 'pure_shampoo_lr0.01'): mean_total_score=1.9731377832392147 (need > 10.0; load from 'local/save/COMP/')
Skipping ('digit_ordered_0_1_75_25_tr100', 'grafted_shampoo_lr0.01'): mean_total_score=2.2762671237332888 (need > 10.0; load from 'local/save/COMP/')
Skipping ('pretrainK1000_tr100_then_0', 'sgd_lr0.01'): mean_total_score=1.4989566698593555 (need > 10.0; load from 'local/save/COMP/')
Skipping ('pretrainK1000_tr100_then_0', 'adagrad_lr0.01'): mean_total_score=0.9563473847529674 (need > 10.0; load from 'local/save/COMP/')
Skipping ('pretrainK1000_tr100_then_0', 'adam_lr0.01'): mean_total_score=1.9650

Expert maps adam_lr0.01…:   0%|          | 0/158 [00:00<?, ?it/s]

Computing representation similarity through time:   0%|          | 0/1739 [00:00<?, ?it/s]

Removing 9 period(s) that do not last until the end of their trial.
Removing: neuron_id            dnn|hidden.6|neuron|0
layer_name                        hidden.6
period_id                                0
start                                 1520
end                                 1541.0
start_iter                            1504
end_iter                              1525
total_length_iter                       21
Name: 14, dtype: object

Removing: neuron_id            dnn|hidden.6|neuron|11
layer_name                         hidden.6
period_id                                 0
start                                  1528
end                                  1534.0
start_iter                             1512
end_iter                               1518
total_length_iter                         6
Name: 15, dtype: object

Removing: neuron_id            dnn|hidden.6|neuron|38
layer_name                         hidden.6
period_id                                 0
start                   

...

Computing representation similarity through time:   0%|          | 0/1740 [00:00<?, ?it/s]

Removing 13 period(s) that do not last until the end of their trial.
Removing: neuron_id            dnn|hidden.4|neuron|16
layer_name                         hidden.4
period_id                                 0
start                                  2840
end                                  2864.0
start_iter                             2811
end_iter                               2835
total_length_iter                        24
Name: 7, dtype: object

Removing: neuron_id            dnn|hidden.4|neuron|4
layer_name                        hidden.4
period_id                                0
start                                  621
end                                  626.0
start_iter                             614
end_iter                               619
total_length_iter                        5
Name: 15, dtype: object

Removing: neuron_id            dnn|hidden.4|neuron|47
layer_name                         hidden.4
period_id                                 0
start                   

Expert maps (extra) pure_shampoo_lr0…:   0%|          | 0/32 [00:00<?, ?it/s]

Computing representation similarity through time:   0%|          | 0/1377 [00:00<?, ?it/s]

Skipping ('pretrain_relabelC_0_1_2_K1000_tr100', 'grafted_shampoo_lr0.01'): mean_total_score=1.5929612148987558 (need > 10.0; load from 'local/save/COMP/')
Computed repr for 7 (eid, oid) pairs; 0 errors


In [9]:
all_results[('pretrain_relabelC_0_1_2_K1000_tr100', 'pure_shampoo_lr0.01')].keys()

dict_keys(['eid', 'mid', 'oid', 'rid', 'impacted_pct', 'repr_per_period'])

In [15]:
all_results[('pretrain_relabelC_0_1_2_K1000_tr100', 'pure_shampoo_lr0.01')]["repr_per_period"]

representation_similarity_factor
neuron_id              period_id                                  
dnn|hidden.2|neuron|1  0                                 -0.024775
dnn|hidden.2|neuron|17 0                                 -0.360168
dnn|hidden.2|neuron|3  0                                 -0.259863
dnn|hidden.2|neuron|40 0                                 -0.125802
dnn|hidden.2|neuron|52 0                                 -0.283153
...                                                            ...
dnn|hidden.8|neuron|57 0                                 -0.392793
dnn|hidden.8|neuron|58 0                                 -0.339955
                       1                                  0.056968
dnn|hidden.8|neuron|61 0                                 -0.206550
dnn|hidden.8|neuron|7  0                                 -0.068608

[84 rows x 1 columns]